# 📱 Spam or Ham? Building an SMS Spam Filter with Logistic Regression

**Episode 4 project · Machine Learning Is Easy**

**The question:** can a model learn to tell spam text messages from normal ones, just by reading the words?

**What you'll practice**
- Turning text into numbers with `CountVectorizer`
- Training a logistic regression classifier
- Reading a confusion matrix, precision and recall
- Why **accuracy can lie** when one class is rare
- Choosing a **threshold**, and looking at the model's mistakes

**Dataset:** [SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection), 5,574 labeled text messages ("ham" = a normal message). Credit: Almeida, T. & Hidalgo, J. (2011), UCI Machine Learning Repository, [doi:10.24432/C5CC84](https://doi.org/10.24432/C5CC84). Licensed under CC BY 4.0.

In [ ]:
# pip install numpy pandas matplotlib scikit-learn
import io
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report,
                             accuracy_score, precision_score, recall_score)

## 1. Load the data

The dataset is one text file where every line is a label (`ham` or `spam`), a tab, and then the message.

The function below downloads the zip from UCI. If the download fails, it reads a local file named `SMSSpamCollection` (download and unzip it from the dataset page).

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

def load_sms():
    try:
        with urllib.request.urlopen(URL, timeout=30) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
        raw = archive.read("SMSSpamCollection")
    except Exception:
        print("Download failed. Reading a local file named SMSSpamCollection instead.")
        with open("SMSSpamCollection", "rb") as f:
            raw = f.read()

    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        text = raw.decode("latin-1")

    text = text.replace("\r\n", "\n")
    rows = [line.split("\t", 1) for line in text.split("\n") if "\t" in line]
    return pd.DataFrame(rows, columns=["label", "text"])

df = load_sms()
print(df.shape)
df.head()

## 2. Look before you model

First: how many messages of each kind? Spam is the **positive class (1)**, because it's the thing we want to catch.

In [ ]:
print(df["label"].value_counts())
print()
print((df["label"].value_counts(normalize=True) * 100).round(1).astype(str) + " %")

Spam is a **minority**. Remember this number: it will matter in a moment.

Let's also look at a few messages, and at message length.

In [ ]:
pd.set_option("display.max_colwidth", 120)

print("A few HAM messages:")
print(df[df["label"] == "ham"]["text"].sample(3, random_state=1).to_string(index=False))
print("\nA few SPAM messages:")
print(df[df["label"] == "spam"]["text"].sample(3, random_state=1).to_string(index=False))

In [ ]:
df["length"] = df["text"].str.len()

for label in ["ham", "spam"]:
    plt.hist(df.loc[df["label"] == label, "length"].clip(upper=300),
             bins=40, alpha=0.6, density=True, label=label)
plt.xlabel("message length (characters, capped at 300)")
plt.ylabel("share of messages")
plt.legend()
plt.title("Do spam and ham differ in length?")
plt.show()

## 3. Split first

Always split **before** training. `stratify=y` keeps the same share of spam in both sets, which is important when one class is rare.

In [ ]:
X = df["text"]
y = (df["label"] == "spam").astype(int)      # 1 = spam, 0 = ham

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(len(X_train), "train /", len(X_test), "test")

## 4. A lazy baseline: always say "ham"

Before building a real model, let's see how a **lazy model** performs. It never looks at the message. It just always predicts the most common class.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_test)

print("Accuracy:        ", round(accuracy_score(y_test, dummy_preds), 3))
print("Spam caught (recall):", round(recall_score(y_test, dummy_preds), 3))

Look at that accuracy. It looks great, yet this model **catches zero spam**. This is the "accuracy lies" problem: when one class is rare, accuracy hides the failure.

Any real model has to beat this on **recall**, not just on accuracy.

## 5. The real model: CountVectorizer + logistic regression

Models need numbers, not words. `CountVectorizer` builds a vocabulary and counts how many times each word appears in each message. Logistic regression then learns which words push a message toward spam.

In [ ]:
model = Pipeline([
    ("vec", CountVectorizer()),
    ("clf", LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)
preds = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, preds), 3))

## 6. Read the confusion matrix

Remember: **positive = spam**.

| | Predicted ham | Predicted spam |
|---|---|---|
| **Actually ham** | TN: delivered, correct | **FP**: a real message sent to the spam folder |
| **Actually spam** | **FN**: spam reached your inbox | TP: spam caught |

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    model, X_test, y_test, display_labels=["ham", "spam"], cmap="Oranges")
plt.title("SMS spam filter")
plt.show()

print(classification_report(y_test, preds, target_names=["ham", "spam"]))

Compare **recall (spam)** with the lazy baseline's zero. Our model actually catches spam now.

Now think about which mistake hurts more:
- **FN** (spam in your inbox) is annoying.
- **FP** (a real message in the spam folder) can mean missing something **important**, like a message from a doctor or a delivery.

For a spam filter, many people care more about **precision**: when the filter says "spam", it should be right.

## 7. Choose your threshold

Logistic regression outputs a **probability of spam**. By default we say "spam" at **0.5 or higher**. Raising the threshold makes the filter more careful: it flags fewer messages, so fewer real ones get caught by mistake.

In [ ]:
proba = model.predict_proba(X_test)[:, 1]

rows = []
for t in [0.3, 0.5, 0.7, 0.9, 0.95]:
    p = (proba >= t).astype(int)
    rows.append({
        "threshold": t,
        "precision (spam)": precision_score(y_test, p, zero_division=0),
        "recall (spam)": recall_score(y_test, p),
        "real messages in spam folder (FP)": int(((p == 1) & (y_test == 0)).sum()),
        "spam reaching inbox (FN)": int(((p == 0) & (y_test == 1)).sum()),
    })

pd.DataFrame(rows).set_index("threshold").round(3)

**The trade-off:** a higher threshold means fewer false alarms (FP) but more spam slipping through (FN). A lower threshold does the opposite. There's no perfect setting. You choose it based on **which mistake costs more** in your situation.

## 8. What did the model learn?

Each word has a **coefficient**. Positive pushes toward spam, negative toward ham.

In [ ]:
vec = model.named_steps["vec"]
clf = model.named_steps["clf"]
coefs = pd.Series(clf.coef_[0], index=vec.get_feature_names_out()).sort_values()

print("Words that push toward HAM:")
print(coefs.head(10).round(2))
print("\nWords that push toward SPAM:")
print(coefs.tail(10)[::-1].round(2))

Do these words make sense to you? A model has no idea what words *mean*. It only learned which words tend to appear in spam.

## 9. Look at the mistakes

Numbers are useful, but reading the actual mistakes teaches you a lot.

In [ ]:
test_df = pd.DataFrame({"text": X_test, "actual": y_test, "predicted": preds})

false_positives = test_df[(test_df["actual"] == 0) & (test_df["predicted"] == 1)]
false_negatives = test_df[(test_df["actual"] == 1) & (test_df["predicted"] == 0)]

print(f"FALSE POSITIVES ({len(false_positives)}): real messages flagged as spam")
print(false_positives["text"].head(5).to_string(index=False))

print(f"\nFALSE NEGATIVES ({len(false_negatives)}): spam that got through")
print(false_negatives["text"].head(5).to_string(index=False))

Why do you think the model got these wrong? Some spam may look like normal chat, and some normal messages may contain "spammy" words. The model only counts words. It doesn't understand context.

## 10. Try your own messages

Edit the list and see what the model thinks.

In [ ]:
my_messages = [
    "Congratulations! You have won a free prize. Text WIN to 80082 to claim now",
    "Are we still meeting for lunch tomorrow?",
    "URGENT! Your mobile number has been selected. Call now to claim your reward",
    "Can you send me the notes from class?",
]

for message, prob in zip(my_messages, model.predict_proba(my_messages)[:, 1]):
    verdict = "SPAM" if prob >= 0.5 else "ham"
    print(f"{verdict:4}  ({prob:.2f})  {message}")

## Recap

- **Logistic regression** outputs a probability, and a **threshold** turns it into a label.
- `CountVectorizer` turns text into numbers; a **Pipeline** keeps the steps together.
- When one class is rare, **accuracy lies**: always compare against a lazy baseline and check **precision and recall**.
- The **confusion matrix** shows *which* mistakes you make, and the two mistakes rarely cost the same.
- Reading the **actual errors** helps you understand the model's limits.

**Try it yourself**
1. Swap `CountVectorizer()` for `TfidfVectorizer()`. Does recall or precision change?
2. Add `ngram_range=(1, 2)` to the vectorizer so it also sees pairs of words.
3. Pick a threshold you'd be happy to ship, and explain why.
4. Next episode we meet decision trees. Can they do better on this data?